In [1]:
def get_data():
    return None

In [2]:
X_train, X_val, X_test, y_train, y_val, y_test = get_data()

TypeError: cannot unpack non-iterable NoneType object

## 1. Rule-Based Baselines
These systems operate on hand-crafted lexicons and linguistic rules without any learning phase. They represent deterministic finite automata that map input tokens to sentiment scores through lookup tables and compositional rules. No optimization objective exists - they're essentially expert systems encoding human domain knowledge.

### 1.1 VADER Sentiment
- Implements a lexicon-based approach with ~7,500 lexical entries
- Each word has a valence score in [-4, +4] validated through crowd-sourcing
- Uses heuristic rules for:
  - Negation handling (sliding window approach)
  - Intensifier detection (degree modifiers like "very", "extremely")
  - Punctuation amplification (exclamation marks boost intensity)
- Final compound score: normalized weighted composite of pos, neu, neg scores
- Time complexity: O(n) where n is text length

In [ ]:
# some code

### 1.2 TextBlob Sentiment
- Built on NLTK's movie review corpus with ~10,000 labeled sentences
- Uses a naive Bayes classifier trained on movie reviews for polarity
- Combines rule-based patterns with statistical models
- Polarity ∈ [-1, 1], Subjectivity ∈ [0, 1]
- Less sophisticated context handling than VADER

In [ ]:
# some code

### 1.3 Custom rule-based approaches (if applicable)
some explenation

In [ ]:
# some code

## 2. Linear Models
These models assume the decision boundary is a hyperplane in the feature space. They learn a weight vector **w** such that the decision function is f(**x**) = **w**^T**x** + b. The key assumption is that sentiment is a linear combination of feature values (typically word frequencies or TF-IDF scores). They optimize convex loss functions, guaranteeing global optima.

In [ ]:
from sklearn.linear_model import Perceptron, LogisticRegression, RidgeClassifier, LinearDiscriminantAnalysis, LinearSVC, PassiveAggressiveClassifier

ImportError: cannot import name 'LinearSVC' from 'sklearn.linear_model' (/opt/miniconda3/envs/nlp/lib/python3.13/site-packages/sklearn/linear_model/__init__.py)

### 2.1 Perceptron
- Learning rule: **w**(t+1) = **w**(t) + η(y - ŷ)**x**
- Mistake-driven online algorithm with guaranteed convergence for linearly separable data
- Updates only on misclassified examples
- No probabilistic interpretation - outputs raw decision function values
- Sensitive to feature scaling and data ordering

In [ ]:
# some code
perceptron = Perceptron(max_iter=500, eta0=0.1, random_state=42).fit(X_train, y_train)


### 2.2 Logistic Regression
- Models P(y=1|**x**) = σ(**w**^T**x**) where σ is the sigmoid function
- Loss function: Cross-entropy (log-likelihood)
- Optimization: Typically L-BFGS or coordinate descent
- Well-calibrated probability estimates
- Maximum likelihood estimation with optional L1/L2 regularization

In [ ]:
# some code
logreg = LogisticRegression(C=1.0, solver='liblinear', max_iter=1000, random_state=42).fit(X_train, y_train)

### 2.3 Ridge Classifier
- Minimizes ||**Xw** - **y**||² + α||**w**||² (L2 penalty)
- Closed-form solution: **w** = (**X**^T**X** + αI)^(-1)**X**^T**y**
- Ridge regression applied to classification via class encoding
- α controls bias-variance tradeoff
- Particularly effective when p >> n (high-dimensional, few samples)

In [ ]:
# some code
ridge = RidgeClassifier(alpha=1.0, random_state=42).fit(X_train, y_train)

### 2.4 Linear Discriminant Analysis (LDA)
- Assumes class-conditional Gaussians with shared covariance: P(**x**|y) ~ N(**μ**_y, **Σ**)
- Decision boundary: **w**^T**x** + b where **w** = **Σ**^(-1)(**μ**₁ - **μ**₀)
- Estimates **μ**_y and **Σ** from training data
- Dimensionality reduction as side effect (projects to C-1 dimensions)
- Can fail with sparse text data due to singularity issues

In [ ]:
# some code
lda = LinearDiscriminantAnalysis(solver='svd').fit(X_train.toarray(), y_train)

### 2.5 Support Vector Machine (Linear)
- Optimization problem: min(½||**w**||² + C∑ξᵢ) subject to yᵢ(**w**^T**x**ᵢ + b) ≥ 1 - ξᵢ
- Finds maximum margin hyperplane
- Only support vectors (examples on margin) determine the model
- Dual formulation allows kernel trick (though we use linear kernel here)
- C parameter trades off margin size vs. training error

In [ ]:
# some code
linear_svc = LinearSVC(C=1.0, max_iter=1000, random_state=42).fit(X_train, y_train)

### 2.6 Passive Aggressive Classifier
- Online algorithm: **w**(t+1) = **w**(t) + τᵢyᵢ**x**ᵢ where τᵢ = loss/||**x**ᵢ||²
- "Passive" on correct predictions, "aggressive" on mistakes
- Bounded updates prevent catastrophic changes
- Suitable for streaming/online learning scenarios
- Sub-linear regret bounds

In [ ]:
# some code
pac = PassiveAggressiveClassifier(C=0.5, max_iter=1000, random_state=42).fit(X_train, y_train)

## 3. Probabilistic Models
Based on Bayesian inference, these models learn P(class|features) using Bayes' theorem: P(y|**x**) ∝ P(**x**|y)P(y). They model the generative process of how text is created given a sentiment class. The "naive" assumption treats features as conditionally independent given the class, drastically reducing the parameter space from exponential to linear complexity.

In [ ]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB

### 3.1 Multinomial Naive Bayes
- Assumes word counts follow multinomial distribution: P(**x**|y) = Multinomial(**x**; **θ**_y)
- Parameter estimation: θ_{yi} = (count(word i in class y) + α) / (total words in class y + α|V|)
- α is Laplace smoothing parameter to handle unseen words
- Works with count vectors (bag-of-words)
- Computational complexity: O(|V| × |C|) for training

In [ ]:
# some code
mnb = MultinomialNB(alpha=1.0).fit(X_train, y_train)

### 3.2 Bernoulli Naive Bayes
- Models word presence/absence: P(xᵢ|y) = θ_{yi}^{xᵢ}(1-θ_{yi})^{(1-xᵢ)}
- Better for short documents where word frequency is less informative
- Parameters: θ_{yi} = P(word i present | class y)
- Works with binary or TF-IDF features after binarization
- More robust to document length variations

In [ ]:
# some code
bnb = BernoulliNB(alpha=1.0).fit(X_train, y_train)

### 3.3 Hidden Markov Models (HMMs)
- State space model: hidden sentiment states generate observable words
- Parameters: π (initial state distribution), A (transition matrix), B (emission matrix)
- Forward-backward algorithm for inference, Baum-Welch for learning
- Viterbi algorithm finds most likely state sequence
- For text classification: separate HMM per class, classify by likelihood

In [ ]:
# some code

## 4. Instance-Based Learning
These are lazy learning algorithms that defer computation until query time. They implement the nearest neighbor rule in feature space, using distance metrics (typically cosine similarity for text) to find similar training instances. No explicit model is built - the training data itself IS the model.

### 4.1 k-Nearest Neighbors (k-NN)
- Distance function: typically cosine similarity for text: cos(**x**, **y**) = **x**^T**y** / (||**x**||||**y**||)
- Classification: majority vote among k nearest neighbors
- Probability estimation: fraction of neighbors in each class
- Curse of dimensionality: performance degrades in very high dimensions
- Space complexity: O(n × d), Time complexity: O(n × d) per query
- Lazy learning: no training phase, all computation at query time

In [ ]:
# some code

## 5. Tree-Based Models
These recursively partition the feature space using axis-aligned splits. Each internal node represents a binary test on a single feature, creating piecewise-constant decision regions. Ensemble methods combine multiple trees through voting (bagging) or sequential error correction (boosting), reducing variance or bias respectively.

In [31]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb

### 5.1 Decision Trees
- Recursive binary splitting using impurity measures:
  - Gini impurity: 1 - ∑P(class i)²
  - Entropy: -∑P(class i)log₂P(class i)
- Greedy algorithm: choose split maximizing information gain
- Stopping criteria: max depth, min samples per leaf, min impurity decrease
- Prone to overfitting due to high variance

In [ ]:
# some code
dtree = DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42).fit(X_train, y_train)

### 5.2 Random Forest
- Bootstrap aggregating (bagging) with feature randomness
- Each tree trained on bootstrap sample with √|features| random features per split
- Prediction: majority vote (classification) or average (regression)
- Out-of-bag error estimation for model validation
- Reduces variance without increasing bias

In [ ]:
# some code
rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42).fit(X_train, y_train)

### 5.3 Gradient Boosting
- Sequential learning: F_m(**x**) = F_{m-1}(**x**) + γ_m h_m(**x**)
- Each new tree h_m fits residuals from previous ensemble
- Learning rate γ controls step size (bias-variance tradeoff)
- Loss function: typically exponential (AdaBoost) or logistic
- High bias, low variance initially → low bias, higher variance over iterations

In [ ]:
gbt = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train)


### 5.4 XGBoost
- Objective: L(θ) = ∑l(yᵢ, ŷᵢ) + ∑Ω(fₖ) where Ω is regularization
- Second-order Taylor approximation of loss function
- Regularization terms: L1 (lasso) + L2 (ridge) on leaf weights
- Advanced features: feature importance, early stopping, handling missing values
- Optimized implementation with parallel tree construction

In [ ]:
xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', learning_rate=0.1, max_depth=4, n_estimators=100, random_state=42).fit(X_train, y_train)

### 5.5 LightGBM
- Gradient-based One-Side Sampling: keeps gradients with large magnitudes
- Exclusive Feature Bundling: bundles sparse features to reduce dimensionality
- Histogram-based algorithm: discrete bins instead of pre-sorted features
- Leaf-wise tree growth vs. level-wise (XGBoost)
- Generally faster training with comparable accuracy

In [ ]:
lgb_clf = lgb.LGBMClassifier(learning_rate=0.1, max_depth=4, n_estimators=100, random_state=42).fit(X_train, y_train)

In [33]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import pandas as pd

In [34]:
path_dataset_train = "../../../2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json"

In [35]:
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
        
    df_raw = pd.DataFrame.from_dict(raw_data, orient="index").reset_index()
    df_raw = df_raw.rename(columns={"index": "id"})
    return df_raw

In [36]:
data_train = load_data(path_dataset_train)

In [37]:
path_hard_train1_2 = "../../../2025 EXIST/evaluation/golds/EXIST2025_training_task1_2_gold_hard.json"

In [38]:
def load_eval(path):
    with open(path, 'r', encoding='utf-8') as f:
        data_from_file = json.load(f)

    df = pd.DataFrame(data_from_file)
    return df

In [39]:
classes_task1_2 = ['NO','DIRECT','REPORTED','JUDGEMENTAL']

In [43]:
# Load soft and hard evaluations from disk
hard_train1_2 = load_eval(path_hard_train1_2)

# Rename target columns
hard_train1_2 = hard_train1_2.rename(columns={'value': 'hard_eval1_2'})

# One-hot encode based on the string values in the column
hard_train1_2_one_hot_df = pd.get_dummies(hard_train1_2['hard_eval1_2']).reindex(columns=classes_task1_2, fill_value=0)
hard_train1_2 = pd.concat([hard_train1_2, hard_train1_2_one_hot_df], axis=1)

# Append to dataset
data_train = pd.merge(data_train, hard_train1_2[['id','NO','DIRECT','REPORTED','JUDGEMENTAL']], on='id', how='left')

In [44]:
data_train_en = data_train[data_train['lang']=='en']

In [71]:
data_train_small = data_train_en[['tweet', 'DIRECT', 'REPORTED', 'JUDGEMENTAL']]

In [72]:
data_train_small.describe()

,tweet,DIRECT,REPORTED,JUDGEMENTAL
count,3260,2620,2620,2620
unique,3260,2,2,2
top,FFS! How about laying the blame on the bastard...,False,False,False
freq,1,2075,2426,2472


In [73]:
df_train = data_train_small.copy()
df_train = df_train[(df_train['DIRECT'].notna()) | (df_train['REPORTED'].notna()) | (df_train['JUDGEMENTAL'].notna())]
df_train = df_train[(df_train['DIRECT']) | (df_train['REPORTED']) | (df_train['JUDGEMENTAL'])]  
df_train = df_train.rename(columns={'tweet': 'text'})

In [74]:
df_train[:10]

,text,DIRECT,REPORTED,JUDGEMENTAL
3661,Writing a uni essay in my local pub with a cof...,False,True,False
3662,@UniversalORL it is 2021 not 1921. I dont appr...,False,True,False
3665,According to a customer I have plenty of time ...,False,True,False
3666,"So only 'blokes' drink beer? Sorry, but if you...",False,True,False
3670,#EverydaySexism means women usually end up in ...,False,False,True
3674,@MarkPaulTimes @colettebrowne #EveryDaySexism ...,True,False,False
3675,@RMatthewsPsyEdu @ITV @jamesmartinchef @Everyd...,True,False,False
3678,Sorry. My feminist rage at the old white man p...,False,True,False
3679,Why do we create comics about sexism in scienc...,False,False,True
3680,you ever get so mad at your ex you send gamerg...,False,False,True


In [51]:
df_big = pd.concat([df.copy() for _ in range(10)], ignore_index=True)

NameError: name 'df' is not defined

In [52]:
len(df_big)

NameError: name 'df_big' is not defined

In [75]:
path_dataset_eval = "../../../2025 EXIST/EXIST 2025 Tweets Dataset/dev/EXIST2025_dev.json"

In [76]:
data_eval = load_data(path_dataset_eval)

In [77]:
path_hard_eval1_2 = "../../../2025 EXIST/evaluation/golds/EXIST2025_dev_task1_2_gold_hard.json"

In [78]:
# Load soft and hard evaluations from disk
hard_eval1_2 = load_eval(path_hard_eval1_2)

# Rename target columns
hard_eval1_2 = hard_eval1_2.rename(columns={'value': 'hard_eval1_2'})

# One-hot encode based on the string values in the column
hard_eval1_2_one_hot_df = pd.get_dummies(hard_eval1_2['hard_eval1_2']).reindex(columns=classes_task1_2, fill_value=0)
hard_eval1_2 = pd.concat([hard_eval1_2, hard_eval1_2_one_hot_df], axis=1)

# Append encoded evaluations to dataset
data_eval = pd.merge(data_eval, hard_eval1_2[['id','NO','DIRECT','REPORTED','JUDGEMENTAL']], on='id', how='left')


In [79]:
data_eval_en = data_eval[data_eval['lang']=='en']

In [80]:
data_eval_small = data_eval_en[['tweet', 'DIRECT', 'REPORTED', 'JUDGEMENTAL']]

In [81]:
data_eval_small.describe()

,tweet,DIRECT,REPORTED,JUDGEMENTAL
count,489,400,400,400
unique,489,2,2,2
top,"@Mike_Fabricant “You should smile more, love. ...",False,False,False
freq,1,313,365,372


In [82]:
df_eval = data_eval_small.copy()
df_eval = df_eval[(df_eval['DIRECT'].notna()) | (df_eval['REPORTED'].notna()) | (df_eval['JUDGEMENTAL'].notna())]
df_eval = df_eval[(df_eval['DIRECT']) | (df_eval['REPORTED']) | (df_eval['JUDGEMENTAL'])]  
df_eval = df_eval.rename(columns={'tweet': 'text'})

In [83]:
df_eval[:10]

,text,DIRECT,REPORTED,JUDGEMENTAL
550,@BBCWomansHour @LabWomenDec @EverydaySexism Sh...,False,True,False
551,#everydaysexism Some man moving my suitcase in...,False,True,False
568,@GoldenSteeler06 @Matthew07219782 @PocketMaxim...,False,True,False
569,I sincerely wish the US was this progressive o...,False,False,True
579,We need to call for A Day Without Women. Where...,True,False,False
581,What a day without women would really do to th...,True,False,False
582,Nearly 10% of students will encounter sexual m...,False,True,False
592,@5heriBr0wn @stinkythinktank @nubbin00_ @Megan...,True,False,False
593,@ParadeofOne @AndiHoppy @AndreaWolper singers....,False,False,True
599,@realwilliamN @Tukatara @hasanthehun @jordanbp...,False,False,True


In [84]:
len(df_eval)

150

In [85]:
df_train.describe()

,text,DIRECT,REPORTED,JUDGEMENTAL
count,887,887,887,887
unique,887,2,2,2
top,Writing a uni essay in my local pub with a cof...,True,False,False
freq,1,545,693,739


In [168]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import time
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
import contractions
from nltk.tokenize import TweetTokenizer

# Download required NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

# Initialize
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
tokenizer = TweetTokenizer()

def expand_contractions(texts: str) -> str:
    return contractions.fix(texts) if isinstance(texts, str) else texts

def reduce_repeated_characters(texts: str) -> str:
    repeat_pattern = re.compile(r'(.)\1{2,}')
    return repeat_pattern.sub(r'\1', texts)

# Negation handling 
def handle_negation(tokens):
    result = []
    negate = False
    for word in tokens:
        if word in ['not', 'no', 'never', "n't"]:
            negate = True
            continue
        if negate:
            result.append('not_' + word)
            negate = False
        else:
            result.append(word)
    return result

# Preprocessing
def preprocess_tweet(text):
    text = text.lower()
    text = expand_contractions(text)  # Expand contractions
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Remove URLs
    text = re.sub(r"@\w+", 'USER', text)                 # Replace mentions
    #text = re.sub(r"@\w+", '', text)  # Remove mentions
    text = re.sub(r"#", '', text)                        # Keep hashtag word
    #text = re.sub(r"#\w+", '', text)  # Remove hashtags
    #text = re.sub(r"[^a-z\s]", '', text)                 # Remove non-letters
    text = reduce_repeated_characters(text)
    
    tokens = tokenizer.tokenize(text)
    tokens = handle_negation(tokens)
    tokens = [lemmatizer.lemmatize(word) for word in tokens 
              if word not in stop_words and len(word) > 1]
    
    return ' '.join(tokens)


# Convert one-hot label columns to a single categorical label
def get_label(row):
    if row['DIRECT']:
        return 'DIRECT'
    elif row['REPORTED']:
        return 'REPORTED'
    elif row['JUDGEMENTAL']:
        return 'JUDGEMENTAL'
    else:
        return 'NONE'  # Should not happen if one label is always True

# Load your data (you already have df_train and df_eval)
df_tr = df_train.copy()
df_ev = df_eval.copy()

# Generate categorical labels
df_tr['label'] = df_tr.apply(get_label, axis=1)
df_ev['label'] = df_ev.apply(get_label, axis=1)

# Preprocess tweets
df_tr['clean_text'] = df_tr['text'].apply(preprocess_tweet)
df_ev['clean_text'] = df_ev['text'].apply(preprocess_tweet)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(ngram_range=(1,1), max_features=10000)
X_train = vectorizer.fit_transform(df_tr['clean_text'])
X_eval = vectorizer.transform(df_ev['clean_text'])

y_train = df_tr['label']
y_eval = df_ev['label']

from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)
X_eval_resampled, y_eval_resampled = ros.fit_resample(X_eval, y_eval)

# Train multiclass logistic regression
model_rf = RandomForestClassifier(
    n_estimators=100,  # Number of trees
    max_depth=None,    # No depth limit, can tune
    random_state=42,
    n_jobs=-1,         # Use all CPU cores
    verbose=1
)

# Gradient Boosting Machine (GBM)
model_gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    #verbose=1
)

# Kernel SVM (RBF kernel)
model_svc = SVC(class_weight='balanced',
    kernel='linear',
    C=100,
    gamma='scale',
    max_iter=100000,
    probability=True,
    #verbose=True
)

# K-Nearest Neighbors
model_knn = KNeighborsClassifier(
    n_neighbors=5,   # Default; can tune
    weights='uniform',  # Or 'distance' to weight neighbors
    n_jobs=-1       # Use all CPUs if available
)

# Decision Tree
model_dt = DecisionTreeClassifier(class_weight='balanced',
    max_depth=None,   # Can tune to prevent overfitting
    random_state=42
)

model_logreg_ovr = LogisticRegression(multi_class='ovr', solver='lbfgs', max_iter=200)

# Quadratic Discriminant Analysis (QDA)
model_qda = QuadraticDiscriminantAnalysis()

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
svc = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)

# Create a VotingClassifier ensemble with soft voting
ensemble = VotingClassifier(
    estimators=[('rf', model_rf), ('gb', model_gb), ('svc', model_svc)],
    voting='soft'  # 'soft' uses predicted probabilities for majority voting
)

model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    C=1.0,
    max_iter=1000,
    random_state=42,
    #verbose=1  # optional, remove if you don't want output
)

start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()
print(f"Training took {end_time - start_time:.2f} seconds.")

# Evaluate
y_pred = model.predict(X_eval)
print(classification_report(y_eval, y_pred))

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Training took 0.04 seconds.
              precision    recall  f1-score   support

      DIRECT       0.60      0.98      0.75        87
 JUDGEMENTAL       0.00      0.00      0.00        28
    REPORTED       0.56      0.14      0.23        35

    accuracy                           0.60       150
   macro avg       0.39      0.37      0.32       150
weighted avg       0.48      0.60      0.49       150



/opt/miniconda3/envs/nlp/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/miniconda3/envs/nlp/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/nlp/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/nlp/lib/python3.13

In [91]:
df_tr['label'] = df_tr.apply(get_label, axis=1)

# Count the number of samples per class
class_counts = df_tr['label'].value_counts()
print(class_counts)

label
DIRECT         545
REPORTED       194
JUDGEMENTAL    148
Name: count, dtype: int64


,test_case,id,hard_eval1_2,NO,DIRECT,REPORTED,JUDGEMENTAL
0,EXIST2025,300002,JUDGEMENTAL,False,False,False,True
1,EXIST2025,300003,NO,True,False,False,False
2,EXIST2025,300004,REPORTED,False,False,True,False
3,EXIST2025,300005,NO,True,False,False,False
4,EXIST2025,300006,NO,True,False,False,False
5,EXIST2025,300007,DIRECT,False,True,False,False
6,EXIST2025,300008,JUDGEMENTAL,False,False,False,True
7,EXIST2025,300009,NO,True,False,False,False
8,EXIST2025,300010,DIRECT,False,True,False,False
9,EXIST2025,300013,DIRECT,False,True,False,False


In [146]:
count = hard_train1_2['JUDGEMENTAL'].sum()
print(f"Number of rows where 'boolean_column' is True: {count}")

Number of rows where 'boolean_column' is True: 376


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, seq_len, embedding_dim=128, lstm_hidden=64, dense_hidden=32):
        super(LSTMSentimentClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, lstm_hidden, batch_first=True, dropout=0.3)
        self.dense = nn.Linear(lstm_hidden, dense_hidden)
        self.dropout = nn.Dropout(0.5)
        self.output = nn.Linear(dense_hidden, 1)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        
        # LSTM output
        lstm_out, (hidden, cell) = self.lstm(embedded)  # lstm_out: (batch_size, seq_len, lstm_hidden)
        
        # Use the last hidden state
        last_hidden = hidden[-1]  # (batch_size, lstm_hidden)
        
        # Dense layers
        dense_out = F.relu(self.dense(last_hidden))  # (batch_size, dense_hidden)
        dropped = self.dropout(dense_out)
        output = torch.sigmoid(self.output(dropped))  # (batch_size, 1)
        
        return output

In [3]:
import torch

counts = torch.tensor([2373, 545, 194, 148], dtype=torch.float32)
class_weights = 1.0 / counts
class_weights = class_weights / class_weights.sum() * len(counts)
class_weights

tensor([0.1190, 0.5180, 1.4553, 1.9077])